In [48]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import joblib
import os

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [49]:
# ==========================================
# Project Paths
# ==========================================

DATA_PATH = r"C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\cleaned"
MODEL_PATH = r"C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\models"
os.makedirs(MODEL_PATH, exist_ok=True)

In [50]:
# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv(
    DATA_PATH + "\\" + "final_dataset.csv"
)

print(df.shape)

df.head()

(515212, 31)


,date,store_id,sku_id,customer_id,quantity,unit_price_x,total_value,channel,discount_pct,sku_name,...,city_y,loyalty_segment,preferred_channel,registration_date,store_id_inventory,stock_on_hand,reorder_point,safety_stock,last_restock_date,snapshot_date
0,2021-01-01,26,1124,2961.0,1,20.08,20.08,Store,15.0,Dairy_Cheese_1124,...,Dubai,Silver,Website,2024-12-19,10,185,70,35,2025-09-27,2025-10-31
1,2021-01-01,38,1088,2507.0,1,17.99,17.99,Website,15.0,Household_Cleaning Supplies_1088,...,Dubai,Gold,Mobileapp,2023-12-17,9,241,96,48,2025-09-21,2025-10-31
2,2021-01-01,2,1093,1252.0,1,7.98,7.98,Store,0.0,Snacks_Chips_1093,...,Dubai,Gold,Mobileapp,2024-11-20,9,289,125,62,2025-09-17,2025-10-31
3,2021-01-01,25,1067,1286.0,1,42.13,42.13,Mobileapp,15.0,Grocery_Cereals_1067,...,Sharjah,Silver,Store,2024-04-02,11,303,116,58,2025-09-27,2025-10-31
4,2021-01-01,2,1043,2507.0,3,19.93,59.79,Mobileapp,15.0,Electronics_Batteries_1043,...,Dubai,Gold,Mobileapp,2023-12-17,9,137,47,23,2025-08-14,2025-10-31


In [51]:
date_columns = [
    col for col in df.columns
    if "date" in col.lower()
]

date_column = date_columns[0]

df[date_column] = pd.to_datetime(df[date_column])

In [52]:
df["Year"] = df[date_column].dt.year

In [53]:
df["Month"] = df[date_column].dt.month

In [54]:
df["Quarter"] = df[date_column].dt.quarter

In [55]:
df["Week"] = df[date_column].dt.isocalendar().week.astype(int)

In [56]:
df["Day"] = df[date_column].dt.day

In [57]:
df["DayOfWeek"] = df[date_column].dt.dayofweek

In [58]:
df["Weekend"] = df["DayOfWeek"].isin([5,6]).astype(int)

In [59]:
if "unit_price_x" in df.columns and "quantity" in df.columns:

    df["Revenue"] = df["unit_price_x"] * df["quantity"]

In [60]:
if "cost" in df.columns:

    df["Profit"] = df["Revenue"] - (df["cost"] * df["quantity"])

In [61]:
if "Profit" in df.columns:

    df["Profit_Margin"] = (
        df["Profit"] /
        df["Revenue"]
    ) * 100

In [62]:
if "stock_quantity" in df.columns:

    df["Inventory_Ratio"] = (

        df["quantity"] /

        (df["stock_quantity"] + 1)

    )

In [63]:
promotion_columns = [

col for col in df.columns

if "promotion" in col.lower()

]

if promotion_columns:

    df["Promotion_Flag"] = (

        df[promotion_columns[0]]

        .notnull()

        .astype(int)

    )

In [64]:
if "quantity" in df.columns:

    df = df.sort_values(date_column)

    df["Lag_1"] = df["quantity"].shift(1)

    df["Lag_7"] = df["quantity"].shift(7)

In [65]:
df["Rolling_Mean_7"] = (

    df["quantity"]

    .rolling(7)

    .mean()

)

In [66]:
df["Rolling_STD_7"] = (

    df["quantity"]

    .rolling(7)

    .std()

)

In [67]:
if "customer_id" in df.columns:

    purchase_count = (

        df.groupby("customer_id")

        .size()

        .reset_index(name="Purchase_Count")

    )

    df = df.merge(

        purchase_count,

        on="customer_id",

        how="left"

    )

In [68]:
if "customer_id" in df.columns:

    lifetime = (

        df.groupby("customer_id")

        ["Revenue"]

        .sum()

        .reset_index(name="Customer_Lifetime_Value")

    )

    df = df.merge(

        lifetime,

        on="customer_id",

        how="left"

    )

In [69]:
if "sku_id" in df.columns:

    freq = (

        df.groupby("sku_id")

        .size()

        .reset_index(name="Sales_Frequency")

    )

    df = df.merge(

        freq,

        on="sku_id",

        how="left"

    )

In [70]:
df.fillna(0, inplace=True)

In [71]:
encoder = LabelEncoder()

categorical = df.select_dtypes(include="object").columns

for col in categorical:

    df[col] = encoder.fit_transform(df[col].astype(str))

In [72]:
numeric_columns = df.select_dtypes(include=np.number).columns

numeric_columns

Index(['store_id', 'sku_id', 'customer_id', 'quantity', 'unit_price_x',
       'total_value', 'channel', 'discount_pct', 'sku_name', 'category',
       'subcategory', 'unit_price_y', 'cost_price', 'brand', 'store_name',
       'city_x', 'store_type', 'opening_date', 'age', 'gender', 'city_y',
       'loyalty_segment', 'preferred_channel', 'registration_date',
       'store_id_inventory', 'stock_on_hand', 'reorder_point', 'safety_stock',
       'last_restock_date', 'snapshot_date', 'Year', 'Month', 'Quarter',
       'Week', 'Day', 'DayOfWeek', 'Weekend', 'Revenue', 'Lag_1', 'Lag_7',
       'Rolling_Mean_7', 'Rolling_STD_7', 'Purchase_Count',
       'Customer_Lifetime_Value', 'Sales_Frequency'],
      dtype='object')

In [73]:
scaler = StandardScaler()

df[numeric_columns] = scaler.fit_transform(

    df[numeric_columns]

)

In [82]:
joblib.dump(

    scaler,

    MODEL_PATH + "\\" + "scaler.pkl"

)

print("Scaler Saved")

Scaler Saved


In [75]:
variance = (

    df[numeric_columns]

    .var()

    .sort_values(ascending=False)

)

variance.head(20)

loyalty_segment            1.000002
Year                       1.000002
Month                      1.000002
city_y                     1.000002
Purchase_Count             1.000002
Lag_1                      1.000002
discount_pct               1.000002
store_type                 1.000002
Customer_Lifetime_Value    1.000002
reorder_point              1.000002
stock_on_hand              1.000002
opening_date               1.000002
unit_price_x               1.000002
Week                       1.000002
unit_price_y               1.000002
sku_id                     1.000002
brand                      1.000002
category                   1.000002
safety_stock               1.000002
last_restock_date          1.000002
dtype: float64

In [76]:
print(df.shape)

df.head()

(515212, 46)


,date,store_id,sku_id,customer_id,quantity,unit_price_x,total_value,channel,discount_pct,sku_name,...,DayOfWeek,Weekend,Revenue,Lag_1,Lag_7,Rolling_Mean_7,Rolling_STD_7,Purchase_Count,Customer_Lifetime_Value,Sales_Frequency
0,2021-01-01,0.036398,0.416027,0.364447,-1.044236,-0.306602,-0.856064,0.289707,1.262380,-1.230732,...,0.386083,-0.72446,-0.856064,-1.754203,-1.754168,-3.470065,-2.896136,-0.575151,-0.575410,0.501105
1,2021-01-01,-0.448451,1.180861,0.001336,-1.044236,-0.519643,-0.942304,0.289707,1.262380,1.142714,...,0.386083,-0.72446,-0.942304,-1.044229,-1.754168,-3.470065,-2.896136,1.738972,1.738972,0.382756
2,2021-01-01,1.421682,0.050992,0.001336,0.375718,0.711878,1.285094,1.073148,-0.613906,-1.100934,...,0.386083,-0.72446,1.285094,-1.044229,-1.754168,-3.470065,-2.896136,1.738972,1.738972,0.423102
3,2021-01-01,0.729040,-0.661694,1.487375,1.085695,-0.044056,0.925495,-1.277176,1.262380,0.549352,...,0.386083,-0.72446,0.925495,0.375721,-1.754168,-3.470065,-2.896136,-0.575223,-0.575139,0.398895
4,2021-01-01,-1.348886,0.485557,-1.631068,-1.044236,0.054955,-0.709702,0.289707,-0.613906,-0.099637,...,0.386083,-0.72446,-0.709702,1.085696,-1.754168,-3.470065,-2.896136,-0.575006,-0.575173,0.495725


In [83]:
import os

output_file = os.path.join(DATA_PATH, "feature_engineered.csv")

df.to_csv(output_file, index=False)

print("feature_engineered.csv Saved")

feature_engineered.csv Saved
